In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
import scipy.stats as stats
from scipy.fft import fft, fftfreq
import seaborn as sns
import sqlite3
from cmap import Colormap

# Working with signal data
So the amplitude and the frequency of the signal are the two main components of many analysis.

Frequency: Mathematically, frequency is the number of waves passing through a fixed point in a single time unit or the number of cycles performed by a body in a single time when it is in a periodic motion.

Amplitude: Amplitude can be defined as the greatest distance travelled by a moving body in a periodic motion in a single time unit or the highest distance of the wave on dips down or rising from its flat surface.

### Load a table from an sqlite database


In [ ]:
settings_DF=pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/settings.csv')
# the ID column of settings_DF is unnamed, so we rename it
#settings_DF.drop(columns={'Unnamed: 0'}, inplace=True)
# load signals and merge them, as they have all the same shape
signal_DFs=[]
for signal in ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']:
    signal_DFs.append(pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/'+signal+'.csv'))
    print(signal_DFs[-1].shape)
# merge the signals
merged_DF=pd.concat(signal_DFs, axis=1)
# remove duplicate ID column
merged_DF=merged_DF.loc[:,~merged_DF.columns.duplicated()]
merged_DF.head(10)


Remove the outliers

In [ ]:
settings_DF.drop(settings_DF.loc[settings_DF['cut_no'].isin([17,94])].index, inplace=True)
merged_DF.drop(merged_DF.loc[merged_DF['cut_no'].isin([17,94])].index, inplace=True)

### Define the time constraints of the data, if not saved explicitly in the data
We need the min and max time, and the "frequency" of data accquisition. Alternatively, we can use the time stamps if existent in our data.

In [ ]:
# define the sampling rate (Hz)
# every 8ms a new sample is taken and there where 9000 samples in the data set for each cut. See Documentation /Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/Mill_DataSheet.pdf for more details.
dt=(8.0/1000.0)
t = np.arange(0, dt*9000, dt)
print(len(t),dt,max(t))

Once we have the time domain and the signal we can perform a Fast-Fourier Transform to get the dominating frequencies in our signal

In [ ]:
# Compute the Fast Fourier Transform (FFT)
def compute_fft(signal,t,dt):
    n = len(t)
    fhat = np.fft.fft(signal, n)                 # Compute the FFT
    psd = fhat * np.conj(fhat) / n          
    freq = (1 / (dt * n)) * np.arange(n)    # frequency array
    idxs_half = np.arange(1, int(np.floor(n / 2)))  # first half index
    
    psd_half = psd[idxs_half]
    freq_half = freq[idxs_half]
    
    return freq_half, psd_half

In [ ]:
# get a list of unique cases
cases = sorted(settings_DF['case'].unique())

# a helper function to create a cut slider for a given case
def get_cut_slider(case):
    cuts = settings_DF[settings_DF['case'] == case]['cut_no'].unique()
    return widgets.IntSlider(min=min(cuts), max=max(cuts), step=1, value=min(cuts), description='Cut No:')

In [ ]:
def plot_fft(cut_no):
    # List of signals to plot
    signal_list = ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
    
    # Create a figure
    fig = plt.figure(figsize=(16, 10))
    
    # Main Gridspec: 1 for table, 1 for plots
    main_gs = fig.add_gridspec(2, 1, height_ratios=[1, 12], hspace=0.2)

    # Table subplot
    ax_table = fig.add_subplot(main_gs[0])
    ax_table.axis('off')

    # Get settings and create table
    cut_settings = settings_DF[settings_DF['cut_no']==cut_no].iloc[0].to_dict()
    table_data = list(cut_settings.items())
    
    table = ax_table.table(cellText=[[v for k, v in table_data]],
                           colLabels=[k for k, v in table_data],
                           loc='center',
                           cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    ax_table.set_title(f'Cut {cut_no} - Settings', fontsize=14)

    # Gridspec for the 2x2 plots
    plot_gs = main_gs[1].subgridspec(2, 2, hspace=0.3, wspace=0.1)
    
    for i, signal in enumerate(signal_list):
        # compute the FFT
        freq, psd = compute_fft(merged_DF[merged_DF['cut_no']==cut_no][signal], t, dt)
        
        # Create a nested GridSpec for signal and FFT plots
        inner_gs = plot_gs[i].subgridspec(2, 1, hspace=0.4)
        
        # Plot the signal
        ax_sig = fig.add_subplot(inner_gs[0])
        ax_sig.plot(t, merged_DF[merged_DF['cut_no']==cut_no][signal], color='orange', linewidth=1)
        ax_sig.set_title(f'{signal} - Time Domain')
        ax_sig.set_xlabel('t in seconds')
        ax_sig.set_xlim(t[0], t[-1])
        
        # Plot the FFT
        ax_fft = fig.add_subplot(inner_gs[1])
        ax_fft.plot(freq, psd.real, color='blue', linewidth=1)
        #ax_fft.set_title('Frequency Domain (PSD)')
        ax_fft.set_xlabel('Frequency in Hz')
        ax_fft.set_ylabel('Amplitude')
        ax_fft.set_xlim(freq[0], freq[-1])
        ax_fft.grid(True)
        
        # Adjust tick labels to prevent overlap
        ax_fft.tick_params(axis='x', labelrotation=45)
        ax_fft.yaxis.set_major_locator(plt.MaxNLocator(5))

    plt.show()

In [ ]:
# interactive plot that depends on the case
case_widget = widgets.Dropdown(options=cases, description='Case:')

def plot_interactive_fft(case):
    cut_slider = get_cut_slider(case)
    widgets.interact(plot_fft, cut_no=cut_slider)

cut_widget = widgets.interactive(plot_interactive_fft, case=case_widget)
display(cut_widget)

## Gabor Transform

A spectrogram is like a photograph or image of a signal
* plots time in Y-axis and frequencies in X-axis.
* explains how the signal strength is distributed in every frequency found in the signal.


In [ ]:
cm_cet_L20 = Colormap('colorcet:CET_L20').to_mpl()  # case insensitive

In [ ]:
def plotSpectral(cut_no):
    # List of signals to plot
    signal_list = ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
    
    # Create a figure
    fig = plt.figure(figsize=(16, 10))
    
    # Main Gridspec: 1 for table, 1 for plots
    main_gs = fig.add_gridspec(2, 1, height_ratios=[1, 12], hspace=0.2)

    # Table subplot
    ax_table = fig.add_subplot(main_gs[0])
    ax_table.axis('off')

    # Get settings and create table
    cut_settings = settings_DF[settings_DF['cut_no']==cut_no].iloc[0].to_dict()
    table_data = list(cut_settings.items())
    
    table = ax_table.table(cellText=[[v for k, v in table_data]],
                           colLabels=[k for k, v in table_data],
                           loc='center',
                           cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    ax_table.set_title(f'Cut {cut_no} - Settings', fontsize=14)

    # Gridspec for the 2x2 plots
    plot_gs = main_gs[1].subgridspec(2, 2, hspace=0.1, wspace=0.1)
    axs = plot_gs.subplots(sharex=True, sharey=True)
    
    for i, signal in enumerate(signal_list):
        row = i // 2
        col = i % 2
        ax = axs[row, col]
        ax.specgram(merged_DF[merged_DF['cut_no']==cut_no][signal], NFFT=128, Fs=1/dt, noverlap=120, cmap='viridis')
        ax.set_title(f'{signal}')
        
    # Add shared labels
    fig.text(0.5, 0.04, 'Time in seconds', ha='center', va='center')
    fig.text(0.06, 0.5, 'Frequency', ha='center', va='center', rotation='vertical')

    plt.show()

In [ ]:
# interactive plot that depends on the case
case_widget2 = widgets.Dropdown(options=cases, description='Case:')
def plot_interactive_spectral(case):
    cut_slider = get_cut_slider(case)
    widgets.interact(plotSpectral, cut_no=cut_slider)

cut_widget2 = widgets.interactive(plot_interactive_spectral, case=case_widget2)
display(cut_widget2)

The `plotSpectral` function in your notebook generates spectrograms, which are powerful tools for analyzing signals like vibration and acoustic emission (AE) data. Here's how to interpret them in the context of your machining data:

### Understanding a Spectrogram

A spectrogram shows how the frequency content of a signal changes over time. Think of it as a series of FFTs stacked together.

*   **X-Axis (Time):** This represents the duration of the signal recording for a specific cut. It shows you *when* something is happening.
*   **Y-Axis (Frequency):** This shows the different frequencies present in the signal. It tells you *what* is happening in terms of oscillations.
*   **Color (Intensity/Power):** The color at any point on the plot indicates the strength or energy of a specific frequency at a specific time. Brighter, "hotter" colors (like yellow in your `viridi` colormap) mean a higher energy/amplitude for that frequency at that moment.

### Interpreting Your Specific Plots

#### 1. Vibration Data (`vib_table`, `vib_spindle`)

Vibration signals measure the physical movement of the machine's table or spindle.

*   **Consistent Horizontal Lines:** Bright, continuous horizontal lines indicate persistent frequencies. These often correspond to the fundamental operating frequencies of the machine, such as the spindle rotation speed or the tool's tooth passing frequency.
*   **Changes in Intensity:** If you see a horizontal band getting brighter over the duration of a cut, it could mean that a particular vibration mode is becoming more pronounced. This can be an early indicator of increasing tool wear, as the tool struggles more to cut the material.
*   **New Frequencies (Chatter):** The appearance of new, strong frequency bands, especially at higher frequencies, can signify the onset of "chatter." Chatter is a harmful self-exciting vibration that leads to poor surface finish and accelerated tool wear.

#### 2. Acoustic Emission Data (`AE_table`, `AE_spindle`)

Acoustic Emission signals are high-frequency stress waves generated by processes like friction, plastic deformation, and material fracture (chip formation). AE is very sensitive to changes at the tool-workpiece interface.

*   **Overall Energy Level:** The general "brightness" of the AE spectrogram is a key indicator. As a tool wears, friction increases, and more energy is required to form a chip. This typically results in a higher overall AE signal energy, making the whole spectrogram appear brighter for a worn tool compared to a fresh one.
*   **Bursts of Energy (Vertical Lines):** Sudden, short-lived events that are bright across a wide range of frequencies (appearing as bright vertical lines) can indicate discrete events like a small part of the tool chipping or breaking.
*   **Frequency Distribution:** While harder to interpret without a baseline, shifts in which frequency bands are most active can point to changes in the cutting mechanism (e.g., a shift from clean shearing to more of a ploughing/rubbing action due to a dull cutting edge).

### In Summary

By using the `plotSpectral_widget`, you can scroll through different cuts (`cut_no`). You should look for trends:
*   **Early Cuts (Fresh Tool):** Expect relatively "clean" spectrograms with distinct, stable frequency bands and lower overall AE energy.
*   **Later Cuts (Worn Tool):** Look for an increase in the overall brightness (energy) of both vibration and AE plots. Watch for new frequency bands appearing in the vibration data (potential chatter) and a general increase in the AE signal level, which are strong indicators of progressive tool wear.

### Get Features from Signals

In [ ]:
# define the features to extract, here is just a subset of the features to start with
FEATURES = ['MIN','MAX','MEAN','RMS','VAR','STD','POWER','PEAK','P2P','CREST FACTOR','SKEW','KURTOSIS',
            'MAX_f','SUM_f','MEAN_f','VAR_f','PEAK_f','SKEW_f','KURTOSIS_f']

# define the function to extract the features from a dataframe
def features_extraction(df): 
    Min=[];Max=[];Mean=[];Rms=[];Var=[];Std=[];Power=[];Peak=[];Skew=[];Kurtosis=[];P2p=[];CrestFactor=[];
    FormFactor=[]; PulseIndicator=[];
    Max_f=[];Sum_f=[];Mean_f=[];Var_f=[];Peak_f=[];Skew_f=[];Kurtosis_f=[]
    
    X = df.values
    ## TIME DOMAIN ##
    # Compute basic stats
    Min.append(np.min(X))
    Max.append(np.max(X))
    Mean.append(np.mean(X))
    # Compute the RMS, variance, standard deviation, power, peak, peak-to-peak, crest factor, skewness, kurtosis, form factor and pulse indicator
    Rms.append(np.sqrt(np.mean(X**2)))
    Var.append(np.var(X))
    Std.append(np.std(X))
    Power.append(np.mean(X**2))
    Peak.append(np.max(np.abs(X)))
    P2p.append(np.ptp(X))
    CrestFactor.append(np.max(np.abs(X))/np.sqrt(np.mean(X**2)))
    Skew.append(stats.skew(X))
    Kurtosis.append(stats.kurtosis(X))
    FormFactor.append(np.sqrt(np.mean(X**2))/np.mean(X))
    PulseIndicator.append(np.max(np.abs(X))/np.mean(X))
    ## FREQ DOMAIN ##
    #Compute the FFT
    n = len(df)
    ft = np.fft.fft(X,n)
    #Compute the single-sided amplitude spectrum and its features
    S = np.abs(ft/n)
    S = S[:n//2]
    Max_f.append(np.max(S))
    Sum_f.append(np.sum(S))
    Mean_f.append(np.mean(S))
    Var_f.append(np.var(S))
    Peak_f.append(np.max(np.abs(S)))
    Skew_f.append(stats.skew(X))
    Kurtosis_f.append(stats.kurtosis(X))

    #Create dataframe from features
    df_features = pd.DataFrame(index = [FEATURES], 
                               data = [Min,Max,Mean,Rms,Var,Std,Power,Peak,P2p,CrestFactor,Skew,Kurtosis,
                                       Max_f,Sum_f,Mean_f,Var_f,Peak_f,Skew_f,Kurtosis_f])
    return df_features

Only rerun it, if you have changed something. It takes some time!
This is a perfect way to introduce some easy to use progress bar

In [ ]:
from tqdm import tqdm # for progress bar

In [ ]:
# create a dataframe to store the features
feature_df=pd.DataFrame(columns=['cut_no','signal'] + FEATURES)
# iterate over all signals
for signal in ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']:
    # iterate over all cuts and extract the features
    print('Extracting features for signal:',signal)
    for i in tqdm(merged_DF.cut_no.unique()):
        # extract the features from the dataframe and transpose it
        df_features = features_extraction(merged_DF[merged_DF['cut_no']==i][signal]).T
        # create a list with the cut number and the signal name
        row=['cut_'+str(i),signal]
        # add the features to the list
        # the features are stored in a numpy array (1,n), so we need to convert it to a list and reshape it to (n,)
        # print('shape', df_features.values.shape)
        row=row+list(df_features.values[0])
        #row=row+df_features.values.reshape(len(df_features),).tolist()
        # add the features to the dataframe
        feature_df.loc[len(feature_df)] = row


In [ ]:
# save the features to a csv file
feature_df.to_csv('./../data/signal_features.csv',index=True)

Let's take a quick look at the features

In [ ]:
# create a boxplot for each signal by feature
def plot_boxplot(signal):
    fig,axes=plt.subplots(1,len(FEATURES),figsize=(24,4))
    for i,feature in enumerate(FEATURES):
        sns.boxplot(x='signal', y=feature, data=feature_df[feature_df['signal']==signal],ax=axes[i])
        axes[i].set_title(feature)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
    plt.show()

# interactive plot
plot_boxplot_widget=interactive(plot_boxplot,signal=['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'])
display(plot_boxplot_widget)

### Pairwise Feature Comparison
Here we create scatter plots to compare each feature from one signal directly against the same feature from a related signal (e.g., `MEAN_table` vs. `MEAN_spindle`). This helps to see how well the features correlate between the two measurement locations.


In [ ]:
# Pivot for vibration signals
vib_pivot = feature_df[feature_df['signal'].isin(['vib_table', 'vib_spindle'])].pivot(index='cut_no', columns='signal', values=FEATURES)

# Determine grid size for subplots
n_features = len(FEATURES)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, feature in enumerate(FEATURES):
    ax = axes[i]
    # Extract the columns for the current feature for both signals
    table_data = vib_pivot[(feature, 'vib_table')]
    spindle_data = vib_pivot[(feature, 'vib_spindle')]
    
    ax.scatter(table_data, spindle_data, alpha=0.5)
    ax.set_title(feature)
    ax.set_xlabel('vib_table')
    ax.set_ylabel('vib_spindle')
    ax.grid(True)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.suptitle('Vibration Features: Table vs. Spindle', y=1.02, fontsize=16)
plt.show()


In [ ]:
# Pivot for AE signals
ae_pivot = feature_df[feature_df['signal'].isin(['AE_table', 'AE_spindle'])].pivot(index='cut_no', columns='signal', values=FEATURES)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, feature in enumerate(FEATURES):
    ax = axes[i]
    # Extract the columns for the current feature for both signals
    table_data = ae_pivot[(feature, 'AE_table')]
    spindle_data = ae_pivot[(feature, 'AE_spindle')]
    
    ax.scatter(table_data, spindle_data, alpha=0.5)
    ax.set_title(feature)
    ax.set_xlabel('AE_table')
    ax.set_ylabel('AE_spindle')
    ax.grid(True)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.suptitle('Acoustic Emission Features: Table vs. Spindle', y=1.02, fontsize=16)
plt.show()


In [ ]:

def create_case_widget():
    # Add 'All' to the list of cases to allow showing all data
    case_options = ['All'] + sorted(settings_DF['case'].unique())
    return widgets.Dropdown(options=case_options, description='Case:', value='All')


In [ ]:

def plot_pairwise_features(case, signal_type, plot_type):
    if case == 'All':
        filtered_feature_df = feature_df
    else:
        # Get the cut numbers for the selected case
        cuts_for_case = settings_DF[settings_DF['case'] == case]['cut_no'].unique()
        # Format cut numbers to match feature_df ('cut_1', 'cut_2', etc.)
        cut_labels = ['cut_' + str(c) for c in cuts_for_case]
        # Filter feature_df
        filtered_feature_df = feature_df[feature_df['cut_no'].isin(cut_labels)]

    if filtered_feature_df.empty:
        print(f"No data available for case: {case}")
        return

    if signal_type == 'Vibration':
        signals_to_plot = ['vib_table', 'vib_spindle']
        title = 'Vibration Features: Table vs. Spindle'
        pivot_df = filtered_feature_df[filtered_feature_df['signal'].isin(signals_to_plot)].pivot(index='cut_no', columns='signal', values=FEATURES)
    elif signal_type == 'Acoustic Emission':
        signals_to_plot = ['AE_table', 'AE_spindle']
        title = 'Acoustic Emission Features: Table vs. Spindle'
        pivot_df = filtered_feature_df[filtered_feature_df['signal'].isin(signals_to_plot)].pivot(index='cut_no', columns='signal', values=FEATURES)
    else:
        return

    if not pivot_df.empty and pivot_df.shape[1] >= len(FEATURES):
        n_features = len(FEATURES)
        n_cols = 5
        n_rows = (n_features + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
        axes = axes.flatten()

        for i, feature in enumerate(FEATURES):
            ax = axes[i]
            table_data = pivot_df.get((feature, signals_to_plot[0]))
            spindle_data = pivot_df.get((feature, signals_to_plot[1]))
            
            if table_data is not None and spindle_data is not None:
                if plot_type == 'Scatter':
                    ax.scatter(table_data, spindle_data, alpha=0.5)
                elif plot_type == 'KDE':
                    sns.kdeplot(x=table_data, y=spindle_data, ax=ax, fill=True)

            ax.set_title(feature, fontsize=8)
            ax.grid(True)

        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        fig.text(0.5, 0.04, 'Table Signal', ha='center', va='center', fontsize=14)
        fig.text(0.06, 0.5, 'Spindle Signal', ha='center', va='center', rotation='vertical', fontsize=14)
        
        plt.tight_layout(rect=[0.08, 0.05, 1, 0.95])
        fig.suptitle(title, fontsize=16)
        plt.show()
    else:
        print(f"Not enough data to plot {signal_type} features for the selected case.")

# Create and display the interactive widget
case_widget = create_case_widget()
signal_widget = widgets.ToggleButtons(
    options=['Vibration', 'Acoustic Emission'],
    description='Signal Type:',
    button_style='',
)
plot_type_widget = widgets.ToggleButtons(
    options=['Scatter', 'KDE'],
    description='Plot Type:',
    button_style='',
)
interactive_plot = widgets.interactive(plot_pairwise_features, case=case_widget, signal_type=signal_widget, plot_type=plot_type_widget)
display(interactive_plot)
